# 02 — Data-quality injection and features
Corrupts the clean sample with **6 types of data-quality error** and records the answer key (`is_dq_anomaly`, `issue_type`). Logic: `src/inject.py` (injections) and `src/features.py` (check features + encoding). Tests: `tests/`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # project root, so `config` and `src` can be imported
import config
import numpy as np
import pandas as pd
from src.prepare import load_clean
from src.inject import make_dirty, inject_missing, inject_balance, inject_numerical, inject_categorical, inject_crossfield
from src.features import reference, build_features

clean = load_clean()
ref = reference(clean)                       # rules learned from the CLEAN data
print(clean.shape)
print('Valid transaction types:', sorted(ref['valid']['type']))
print('City -> state:', ref['city_state'])

(50000, 22)
Valid transaction types: ['BPAY', 'CASH_OUT', 'DEBIT', 'EFTPOS', 'NPP', 'OSKO', 'TRANSFER']
City -> state: {'Adelaide': 'SA', 'Brisbane': 'QLD', 'Melbourne': 'VIC', 'Other': 'NSW', 'Perth': 'WA', 'Sydney': 'NSW'}


## 1. See each injection on 5 real rows (values in quotes so spaces are visible)

In [2]:
cols = ['type', 'category', 'payment_method', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'city', 'state', 'hour']
demo = clean.head(5).copy()
rng = np.random.default_rng(1)
for name, func in [('missing', inject_missing), ('balance', inject_balance), ('numerical', inject_numerical),
                   ('categorical', inject_categorical), ('crossfield', inject_crossfield)]:
    after = demo.copy()
    func(after, after.index, rng)
    changed = [c for c in cols if not after[c].equals(demo[c])]
    print(f'===== {name}: changed columns -> {changed}')
    print(pd.concat({'before': demo[changed], 'after': after[changed]}, axis=1).map(repr), '\n')

===== missing: changed columns -> ['category', 'amount', 'newbalanceOrig', 'city']
         before                                                       \
       category              amount      newbalanceOrig         city   
0        'Food'   217.1611013905674   201651.8796825831      'Perth'   
1  'Recreation'  255.02495758190383   224927.4863911116      'Other'   
2       'Other'  200.73488707408336  256916.92176927387   'Brisbane'   
3       'Other'   181.4146973650994    334979.034666753  'Melbourne'   
4     'Housing'   5686.387323935304  187394.00394327735     'Sydney'   

          after                                                      
       category              amount      newbalanceOrig        city  
0        'Food'   217.1611013905674                 nan     'Perth'  
1  'Recreation'  255.02495758190383                 nan     'Other'  
2           nan  200.73488707408336  256916.92176927387  'Brisbane'  
3       'Other'   181.4146973650994    334979.034666753       

| Issue | Injection | Detected by |
| --- | --- | --- |
| missing | blanks one of amount, oldbalanceOrg, newbalanceOrig, category, city | `*_missing` |
| duplicate | appends an exact copy of an untouched row (only the copy is labelled) | `is_repeat` |
| balance | adds €10–1,000 (±) to the closing balance | `abs_balance_diff` |
| numerical | amount × 10, × 100 or negative; balances keep the true values | `abs_balance_diff`, `amount_log`, `amount_negative` |
| categorical | spelling variant of type / category / payment method | `*_invalid` |
| crossfield | city in the wrong state, or hour not matching the timestamp | `city_state_mismatch`, `hour_mismatch` |

## 2. Build the dirty datasets: 3 severities × 3 seeds

In [3]:
summary = []
for rate in config.RATES:
    for seed in config.SEEDS:
        tag = config.dataset_tag(rate, seed)
        dirty = make_dirty(clean, rate, seed)
        X, y, issue = build_features(dirty, ref)
        dirty.to_csv(config.PROCESSED / f'dirty_{tag}.csv.gz', index=False)
        X.assign(is_dq_anomaly=y, issue_type=issue).to_csv(config.PROCESSED / f'features_{tag}.csv.gz', index=False)
        summary.append({'dataset': tag, 'rows': len(dirty), 'anomalies': int(y.sum()), 'anomaly_%': round(y.mean() * 100, 2),
                        'per_issue': int((issue == 'missing').sum()), 'features': X.shape[1]})
pd.DataFrame(summary)

,dataset,rows,anomalies,anomaly_%,per_issue,features
0,r01_s42,50083,498,0.99,83,63
1,r01_s7,50083,498,0.99,83,63
2,r01_s123,50083,498,0.99,83,63
3,r05_s42,50417,2502,4.96,417,63
4,r05_s7,50417,2502,4.96,417,63
5,r05_s123,50417,2502,4.96,417,63
6,r10_s42,50833,4998,9.83,833,63
7,r10_s7,50833,4998,9.83,833,63
8,r10_s123,50833,4998,9.83,833,63


## 3. Verify: each check fires on its own issue and never on clean rows

In [4]:
f = pd.read_csv(config.PROCESSED / 'features_r05_s42.csv.gz')
checks = pd.DataFrame({
    'issue_type': f['issue_type'],
    'missing_flag': f.filter(like='_missing').sum(axis=1) > 0,
    'is_repeat': f['is_repeat'] == 1,
    'balance_diff': f['abs_balance_diff'] > 0.01,
    'negative_amount': f['amount_negative'] == 1,
    'invalid_category': f.filter(like='_invalid').sum(axis=1) > 0,
    'crossfield_mismatch': (f['city_state_mismatch'] + f['hour_mismatch']) > 0})
checks.groupby('issue_type').mean().round(3)

,missing_flag,is_repeat,balance_diff,negative_amount,invalid_category,crossfield_mismatch
issue_type,,,,,,
balance,0.0,0.0,1.0,0.000,0.0,0.0
categorical,0.0,0.0,0.0,0.000,1.0,0.0
crossfield,0.0,0.0,0.0,0.000,0.0,1.0
duplicate,0.0,1.0,0.0,0.000,0.0,0.0
missing,1.0,0.0,0.0,0.000,0.0,0.0
none,0.0,0.0,0.0,0.000,0.0,0.0
numerical,0.0,0.0,1.0,0.307,0.0,0.0


**Reading this table:** each issue triggers its own check in 100% of rows, and no check fires on clean rows. Numerical errors also trigger `balance_diff` (the balances keep the true amount), and about a third of them have a negative amount (the "wrong sign" variant).

**Why this matters:** the labels are exact, so every model result and every explanation can be checked against the true cause.


In [5]:
# A duplicate next to its original
d = pd.read_csv(config.PROCESSED / 'dirty_r05_s42.csv.gz')
key = ['nameOrig', 'nameDest', 'amount', 'timestamp']
d.merge(d[d['issue_type'] == 'duplicate'].head(2)[key], on=key)[key + ['type', 'is_dq_anomaly', 'issue_type']]

,nameOrig,nameDest,amount,timestamp,type,is_dq_anomaly,issue_type
0,C7578,C7492,133.914613,2026-03-26 13:33:07.636449,TRANSFER,0,none
1,C1017,C630,160.491467,2026-01-05 09:42:04.636449,DEBIT,0,none
2,C7578,C7492,133.914613,2026-03-26 13:33:07.636449,TRANSFER,1,duplicate
3,C1017,C630,160.491467,2026-01-05 09:42:04.636449,DEBIT,1,duplicate


In [6]:
# The clean data itself triggers no checks (proof the checks don't create false alarms)
Xc, _, _ = build_features(clean.assign(is_dq_anomaly=0, issue_type='none'), ref)
print('Checks firing on clean data:', int(Xc.filter(regex='missing|repeat|invalid|mismatch').values.sum()),
      '| balance errors:', int((Xc['abs_balance_diff'] > 0.01).sum()))
print('Feature columns:', Xc.shape[1])
print(list(Xc.columns))

Checks firing on clean data: 0 | balance errors: 0
Feature columns: 63
['amount_missing', 'oldbalanceOrg_missing', 'newbalanceOrig_missing', 'category_missing', 'city_missing', 'is_repeat', 'abs_balance_diff', 'amount_negative', 'amount_log', 'type_invalid', 'category_invalid', 'payment_method_invalid', 'city_state_mismatch', 'hour_mismatch', 'oldbalanceOrg', 'newbalanceOrig', 'hour', 'day_of_week', 'day_of_month', 'month', 'amount_vs_average', 'customer_risk_score', 'sender_degree', 'type_BPAY', 'type_CASH_OUT', 'type_DEBIT', 'type_EFTPOS', 'type_NPP', 'type_OSKO', 'type_TRANSFER', 'category_Education', 'category_Food', 'category_Healthcare', 'category_Housing', 'category_Other', 'category_Recreation', 'category_Shell Company', 'category_Transport', 'category_Utilities', 'payment_method_BSB_Account', 'payment_method_CardNumber', 'payment_method_PayID', 'city_Adelaide', 'city_Brisbane', 'city_Melbourne', 'city_Other', 'city_Perth', 'city_Sydney', 'state_NSW', 'state_QLD', 'state_SA', '